# Praca magisterska - Eksperymenty i wyniki
### Tytuł pracy: Metody sztucznej inteligencji w implementacji bohatera niezależnego w grach komputerowych
### Autor: Kamil Wyżgoł

## Spis treści:

- [1. Wprowadzenie](#wprowadzenie)
- [2. Ustawienia i parametry](#ustawienia)
- [3. Środowisko badawcze](#srodowisko-badawcze)
  - [3.1 Opis środowiska gry komputerowej](#opis-srodowiska)
  - [3.2 Agent bazowy](#agent-bazowy)
- [4. Eksperymenty](#eksperymenty)
- [5. Przykładowe pojedynki (filmy)](#filmy-pojedynkow)


In [1]:
import pandas as pd

from copy import deepcopy
from itertools import combinations
from pathlib import Path
from src.agents.agents import AgentNN
from src.agents.base_agent import DecisionTreeAgent
from src.duels.duels import duels
from src.training.train_dqn import train_dqn
from src.training.train_double_dqn import train_double_dqn
from src.training.train_dueling_dqn import train_dueling_dqn
from src.training.train_per_dqn import train_per_dqn
from src.training.train_noisy_dqn import train_noisy_dqn
from src.training.train_distributional_dqn import train_distributional_dqn
from src.training.train_reinforce import train_reinforce
from src.training.train_a2c import train_a2c
from src.training.train_ppo import train_ppo
from src.training.train_nstep_dqn import train_nstep_dqn
from src.training.train_rainbow_dqn import train_rainbow_dqn
from src.training.parallel_training import make_training_job, run_parallel_training_jobs
from src.utils.data_analysis import calculate_auc_reward_test_basic, sort_groups_by_mean_desc, test_kruskal, \
    test_mannwhitneyu
from src.utils.game_time import TimeManager, GTime
from src.utils.settings import GlobalSettings
from src.utils.enums import ActivationFunction, SignificantStatus
from src.utils.visualization import show_training_summary, training_compare_table, show_heatmap_winrate_train, \
    show_heatmap_winrate_test, show_heatmap_winrate_diff, show_heatmap_episode_train, show_heatmap_episode_test

<a id="wprowadzenie"></a>
## 1. Wprowadzenie

<p>Celem pracy jest analiza i ocena różnych metod sztucznej inteligencji w kontekście implementacji systemu sterowania zachowaniem postaci niezależnej w kompetytywnej grze komputerowej. Badaniu zostaną poddane algorytmy uczenia ze wzmocnieniem (ang. reinforcement learning), w tym algorytmy oparte na tzw. Q-wartości (ang. Q-learning) i algorytmy oparte na gradiencie polityki (ang. policy gradient). W ramach pracy opracowane zostanie również parametryzowane środowisko treningowe i testowe, które zostaną wykorzystane do treningu oraz ewaluacji modeli sztucznej inteligencji oraz będą symulować warunki typowe dla kompetytywnej gry komputerowej.</p>
<p>Oczekuje się, że przeprowadzone w ramach pracy badania pozwolą na ocenę efektywności wspomnianych metod w kontekście implementacji systemu sterowania zachowaniem postaci niezależnej w środowiskach gier o charakterze rywalizacyjnym oraz umożliwią sformułowanie wniosków dotyczących ich zastosowania w innych implementacjach o zbliżonych założeniach.</p>
<p>Niniejszy notatnik stanowi część badawczą (eksperymentalną) pracy magisterskiej</p>

<a id="ustawienia"></a>
## 2. Ustawienia i parametry wykorzystywane przez środowisko oraz algorytmy RL

In [2]:
# klasa do zarządzania czasem symulacji i konwersji czasu rzeczywistego na kroki środowiska
tm = TimeManager()

# ustawienie czasu reakcji agentów na zbieżny z ludzkim
tm.interval = GTime.milliseconds(250)

# ustawienie maksymalnego czasu trwania epizodu
tm.episode_time_limit = GTime.minutes(10)

In [3]:
settings = GlobalSettings()

# ============================= ŚRODOWISKO =============================

# --- rozmiar mapy ---
settings.width = 15
settings.height = 15

# --- tworzenie mapy ---

# ile procent obiektów ma być danego typu
settings.walls_ratio = 0.16
settings.turret_ratio = 0.04
settings.trap_ratio = 0.05

# określenie czy seedy są powtarzane z zamienionymi pozycjami startowymi graczy
settings.add_reversed_positions = True

# --- czas ---
settings.tm = tm
settings.episode_max_steps = tm.get_episode_max_steps()
settings.eval_after = 200_000
settings.eval_episodes_training_base_agent = 5
settings.eval_episodes_training_previous_agent = 5
settings.eval_episodes_test_base_agent = 5
settings.eval_episodes_test_previous_agent = 5
settings.total_max_training_steps = 10_000_000

# --- cooldowny ---
settings.move_cooldown = tm.duration(GTime.seconds(0.5))
settings.melee_cooldown = tm.duration(GTime.seconds(0.75))
settings.ranged_cooldown = tm.duration(GTime.seconds(1.25))
settings.dash_cooldown = tm.duration(GTime.seconds(2.0))
settings.absolute_defence_cooldown = tm.duration(GTime.seconds(12.0))
settings.attack_boost_cooldown = tm.duration(GTime.seconds(10.0))

# --- czasy trwania efektów ---
settings.absolute_defence_duration = tm.duration(GTime.seconds(2.0))
settings.attack_boost_duration = tm.duration(GTime.seconds(4.0))
settings.slowed_duration = tm.duration(GTime.seconds(3.0))

# --- mnożniki efektów ---
settings.attack_boost_multiplier = 1.5
settings.slowed_cooldown_multiplier = 1.5

# --- zasoby ---
settings.max_hp = 100
settings.max_strength = 100
settings.max_mana = 100
settings.max_stamina = 100

# --- inicjalizacja zasobów ---
settings.initial_hp = settings.max_hp
settings.initial_strength = settings.max_strength
settings.initial_mana = settings.max_mana
settings.initial_stamina = settings.max_stamina

# --- regeneracja zasobów ---
settings.strength_regen = tm.value_per_second(8.0)
settings.mana_regen = tm.value_per_second(5.0)
settings.stamina_regen = tm.value_per_second(7.0)

# --- koszty akcji ---
settings.melee_strength_cost = 10.0
settings.ranged_mana_cost = 20
settings.dash_stamina_cost = 20
settings.absolute_defence_mana_cost = 40
settings.attack_boost_stamina_cost = 40

# --- obrażenia ---
settings.melee_damage = 3.0
settings.projectile_damage = 7.0
settings.turret_projectile_damage = 3.0
settings.trap_damage = tm.value_per_second(2.0)

# --- efekty trafień ---
settings.melee_applies_slow = True
settings.projectile_applies_slow = True

# --- zasada działania pocisków/wieżyczek ---
settings.projectile_move_cooldown = tm.duration(GTime.seconds(0.25))
settings.turret_shoot_interval = tm.duration(GTime.seconds(4.0))

# --- nagrody ---
settings.damage_dealt_reward_scale = 1.0
settings.damage_taken_penalty_scale = 1.0
settings.invalid_action_penalty = -0.001
settings.step_penalty = -0.001

settings.win_reward_bonus = 25.0
settings.time_reward_bonus = 25.0

# ================================== GUI ==================================

settings.empty_tile_color = (120, 200, 120)
settings.wall_tile_color = (55, 55, 55)
settings.trap_tile_color = (190, 40, 40)
settings.turret_tile_color = (245, 180, 40)

settings.player_1_color = (75, 119, 209)
settings.player_2_color = (234, 51, 35)

settings.hp_bar_color = (220, 40, 40)
settings.strength_bar_color = (230, 210, 40)
settings.mana_bar_color = (40, 110, 230)
settings.stamina_bar_color = (40, 190, 80)

# ==================== Drzewo decyzyjne (agent bazowy) ====================

settings.random_movement_chance = 0.3

# ============================ Proces treningu ============================

settings.advanced_training_possible = True
settings.advanced_training_winrate_threshold = 0.7
settings.advanced_training_required_evaluations = 3
settings.advanced_model_chance = 0.7

# ================================= AI/RL =================================

# --- shared ---
settings.gamma = 0.99
settings.learning_rate = 1e-4
settings.force_cpu_training = True
settings.torch_num_threads = 1

settings.hidden_layers = 3
settings.hidden_dims = 128
settings.f_activation = ActivationFunction.RELU

# --- value-based shared ---
settings.replay_buffer_size = 100_000
settings.batch_size = 64
settings.warmup_steps = 1_000

settings.target_update_freq = 1_000
settings.optimize_every = 4

settings.epsilon_start = 1.0
settings.epsilon_end = 0.05
settings.epsilon_decay = 500_000

# --- N-step ---
settings.n_step = 1

# --- PER ---
settings.per_alpha = 0.6
settings.per_beta_start = 0.4
settings.per_beta_end = 1.0
settings.per_epsilon = 1e-6

# --- Noisy ---
settings.sigma_init = 0.5

# --- Distributional ---
settings.num_atoms = 51
settings.v_min = -150.0
settings.v_max = 150.0

# --- policy-gradient shared ---
settings.rollout_steps = 2048
settings.entropy_coef = 0.01

# --- actor-critic shared ---
settings.value_loss_coef = 0.5
settings.gae_lambda = 0.95

# --- PPO ---
settings.clip_epsilon = 0.2
settings.ppo_epochs = 4
settings.ppo_batch_size = 64

<a id="srodowisko-badawcze"></a>
## 3. Środowisko badawcze
Środowisko badawcze zostało przygotowane specjalnie na potrzeby przeprowadzanych eksperymentów.

W jego skład wchodzą:
- środowisko będące symulacją kompetytywnej gry komputerowej,
- implementacje algorytmów uczenia ze wzmocnieniem,
- agent bazowy stanowiący wspólny punkt odniesienia,
- moduły odpowiedzialne za trening i ewaluację modeli RL,
- moduły odpowiedzialne za wizualizację wyników oraz renderowanie pojedynków.

<a id="opis-srodowiska"></a>
### 3.1 Opis środowiska gry komputerowej

Środowisko gry komputerowej zostało zaimplementowane jako uproszczona, dwuwymiarowa arena walk, w której dwóch równorzędnych graczy rywalizuje ze sobą na planszy o dyskretnym układzie pól.

Każdy z graczy posiada identyczny zestaw akcji, zasobów oraz mechanik rozgrywki, co pozwala na możliwie sprawiedliwe porównywanie różnych metod sztucznej inteligencji.

Środowisko zostało zaprojektowane w sposób umożliwiający:
- prowadzenie treningu agentów RL,
- ewaluację modeli w powtarzalnych warunkach (środowiska parametryzowane za pomocą wartości tzw. seedów),
- wizualizację przebiegu treningów,
- odpowiednie porównanie otrzymanych wyników.

Poniżej przedstawiono najważniejsze założenia i aspekty środowiska.

#### Akcje gracza (agenta)

Agent może wykonywać następujące rodzaje akcji:
- ruch w czterech kierunkach,
- atak wręcz w czterech kierunkach,
- atak dystansowy w czterech kierunkach,
- zryw (dash) w czterech kierunkach,
- aktywację obrony absolutnej,
- aktywację wzmocnienia ataku.

#### Zasoby gracza

Każdy z graczy posiada zestaw zasobów wpływających na możliwość wykonywania określonych akcji:
- Zdrowie - jego całkowita utrata powoduje przegraną przez danego gracza, nie odnawia się w trakcie.
- Siła - zasób odpowiedzialny za walkę wręcz.
- Magia - zasób odpowiedzialny za ataki dalekodystansowe oraz umiejętność obrony absolutnej.
- Wytrzymałość - zasób odpowiedzialny za zryw oraz wzmocnienie ataku.

#### Stany specjalne postaci:
- Obrona absolutna - uniemożliwia ponoszenie jakichkolwiek obrażeń przez określony czas
- Wzmocnienie ataku - zwiększenie wartości zadawanych obrażeń przez atak wręcz (np. x1.5) na określony czas
- Spowolnienie - wydłużenie czasu odnowienia możliwości ruchu (np. x1.5) na określony czas

#### Obiekty składowe środowiska

Na planszy mogą występować różne rodzaje obiektów:
- Postać - jedna z dwóch postaci wchodzących w interakcje ze środowiskiem.
- Pusta przestrzeń - pole możliwe do przemierzania przez postać.
- Przeszkoda - blokada uniemożliwiająca ruch lub zryw w danym kierunku.
- Pułapka - pole zadające obrażenia postaci, która się na nim znajdzie.
- Wieżyczka - obiekt, który co jakiś czas wystrzeliwuje pociski oraz uniemożliwia poruszanie w danym miejscu (podobnie jak przeszkoda).
- Pocisk - obiekt poruszający się w danym kierunku dopóki nie trafi na jakąś przeszkodę lub gracza (gdy to nastąpi jest usuwany ze środowiska). Po trafieniu gracza zadaje mu obrażenia oraz nadaje stan spowolnienia. Może być wystrzelony przez gracza lub przez wieżyczkę. Gdy dwa pociski się zderzą to oba znikają.

#### Wizualizacja środowiska

Na potrzeby analizy zachowań agentów przygotowano system renderowania pojedynków umożliwiający generowanie filmów przedstawiających przebieg rozgrywki.

Przykładowe filmy pojedynków zostały umieszczone w końcowej części notatnika.

Poniżej przedstawiono przykładową wizualizację środowiska.

<img src="images/przykład środowiska.png" />

<a id="agent-bazowy"></a>
## 3.2. Agent bazowy (klasa DecisionTreeAgent)
Agent bazowy został zaimplementowany na podstawie ręcznie przygotowanego drzewa decyzyjnego.

Celem agenta nie jest osiągnięcie maksymalnej skuteczności, ale stworzenie stosunkowo przewidywalnego i łatwego do interpretacji przeciwnika stanowiącego punkt odniesienia dla metod uczenia ze wzmocnieniem.

Zachowanie agenta opiera się na zestawie prostych reguł i priorytetów decyzyjnych. W większości sytuacji agent zachowuje się deterministycznie, jednak w wybranych przypadkach może wykonywać częściowo losowe ruchy, co ogranicza pełną powtarzalność zachowania i utrudnia nadmierne dopasowanie modeli RL do jednego schematu działania.

#### Lista priorytetów agenta bazowego obejmuje:
- atak wręcz, jeżeli przeciwnik znajduje się na sąsiednim polu i akcja ataku wręcz jest dostępna,
- brak akcji, jeżeli przeciwnik znajduje się na sąsiednim polu, ale atak wręcz jest chwilowo niedostępny,
- atak dystansowy, jeżeli przeciwnik znajduje się w tej samej linii i akcja ataku dystansowego jest dostępna,
- poruszanie się w kierunku przeciwnika z wykorzystaniem algorytmu BFS, jeżeli bezpośredni atak nie jest możliwy,
- częściowo losowy wybór ruchu zamiast ruchu BFS z prawdopodobieństwem określonym przez parametr `random_movement_chance`.

Poniżej przedstawiono uproszczony diagram decyzyjny opisujący zachowanie agenta bazowego.

<img src="images/agent bazowy.png" />

<a id="eksperymenty"></a>
## 4. Eksperymenty

Celem eksperymentów było porównanie różnych metod sztucznej inteligencji w kontekście sterowania bohaterem niezależnym w środowisku rywalizacyjnej gry komputerowej. Badania obejmowały zarówno analizę procesu treningu modeli, jak i ocenę skuteczności agentów podczas bezpośrednich pojedynków.

Poszczególne eksperymenty koncentrowały się między innymi na:
1) wpływie architektury sieci neuronowej (funkcje aktywacji, ilość warstw i neuronów ukrytych) na skuteczność treningu,
2) sprawdzeniu jak parametr *n* wpływa na wyniki dla algorytmu N-step DQN,
3) porównaniu wariantów algorytmu DQN (od zwykłego DQN, poprzez różne modyfikacje, aż do Rainbow DQN będącego połączeniem wszystkich wariantów).
4) porównaniu metod policy-gradient (REINFORCE, A2C, PPO),
5) analizie wyników pojedynków otrzymanych agentów podczas wzajemnych zmagań.

W celu zapewnienia możliwie powtarzalnych warunków eksperymentalnych wykorzystano oddzielne pule seedów dla środowisk treningowych oraz testowych.

Wszystkie eksperymenty wykonywano wielokrotnie, a otrzymane wyniki agregowano oraz analizowano statystycznie.

Podczas treningu dokonywano okresowej ewaluacji modeli, a po osiągnięciu dostatecznej skuteczności przeciwko bazowego agentowi trening miał możliwość częściowego przejścia w tryb system self-play, w którym agent po osiągnięciu odpowiedniego poziomu skuteczności mógł trenować również przeciwko poprzedniej wersji własnego modelu.

In [4]:
# Seedy eksperymentów
TRAINING_SEEDS = {
    "r1": ["train_1_1", "train_1_2", "train_1_3", "train_1_4", "train_1_5"],
    "r2": ["train_2_1", "train_2_2", "train_2_3", "train_2_4", "train_2_5"],
    "r3": ["train_3_1", "train_3_2", "train_3_3", "train_3_4", "train_3_5"],
    "r4": ["train_4_1", "train_4_2", "train_4_3", "train_4_4", "train_4_5"],
}

TEST_SEEDS = ["test_1", "test_2", "test_3", "test_4", "test_5"]

DUEL_SEEDS = ["duel_1", "duel_2", "duel_3", "duel_4", "duel_5" ]


# Liczba powtórzeń eksperymentów
NUM_REPEATS = 3

In [5]:
from src.utils.data_analysis import (
    calculate_mean_episode_length_test_basic,
    calculate_mean_episode_length_test_previous,
    calculate_mean_episode_length_train_basic,
    calculate_mean_episode_length_train_previous,
    calculate_mean_mean_reward_test_basic,
    calculate_mean_mean_reward_test_previous,
    calculate_mean_mean_reward_train_basic,
    calculate_mean_mean_reward_train_previous,
    calculate_mean_winrate_test_basic,
    calculate_mean_winrate_test_previous,
    calculate_mean_winrate_train_basic,
    calculate_mean_winrate_train_previous,
)

RESULTS_DIR = Path("outputs/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS_DIR = Path("outputs/settings")
SETTINGS_DIR.mkdir(parents=True, exist_ok=True)

PARALLEL_WORKERS = 7

def save_results(df: pd.DataFrame, experiment_name: str) -> Path:
    path = RESULTS_DIR / f"{experiment_name}.csv"
    df.to_csv(path, index=False)
    return path

def load_results(experiment_name: str) -> pd.DataFrame:
    return pd.read_csv(RESULTS_DIR / f"{experiment_name}.csv")

def run_training_jobs(jobs: list[dict]):
    return run_parallel_training_jobs(jobs, max_workers=PARALLEL_WORKERS)

def collect_metric_results(method_key: str, method_name: str, calculate_function):
    return (
        method_name,
        [
            calculate_function(load_results(f"{method_key}_r{run_number}"))
            for run_number in range(1, NUM_REPEATS + 1)
        ],
    )

def build_metric_results(variants: list[tuple[str, str, object]], calculate_function):
    return [
        collect_metric_results(method_key, method_name, calculate_function)
        for method_key, method_name, _ in variants
    ]


<a id="eksperyment-1"></a>
### Eksperyment 1 - Dobór parametru n dla N-step DQN


#### Opis:
Celem eksperymentu jest porównanie bazowego DQN z wariantem N-step DQN dla wartości `n = 2`, `n = 4` oraz `n = 6`. Bazowy DQN jest traktowany jako przypadek `n = 1`. Najlepsza wartość `n` jest wybierana na podstawie średniej wartości średniej nagrody w środowisku treningowym przeciwko modelowi bazowemu.


In [6]:
n_2_settings = deepcopy(settings)
n_2_settings.n_step = 2

n_4_settings = deepcopy(settings)
n_4_settings.n_step = 4

n_6_settings = deepcopy(settings)
n_6_settings.n_step = 6

TRAINING_1_VARIANTS = [
    ("dqn", "DQN", train_dqn),
    ("nstep_dqn_n2", "N-step DQN (n=2)", train_nstep_dqn),
    ("nstep_dqn_n4", "N-step DQN (n=4)", train_nstep_dqn),
    ("nstep_dqn_n6", "N-step DQN (n=6)", train_nstep_dqn),
]

TRAINING_1_SETTINGS = {
    "dqn": settings,
    "nstep_dqn_n2": n_2_settings,
    "nstep_dqn_n4": n_4_settings,
    "nstep_dqn_n6": n_6_settings,
}

TRAINING_1_N_VALUES = {
    "dqn": 1,
    "nstep_dqn_n2": 2,
    "nstep_dqn_n4": 4,
    "nstep_dqn_n6": 6,
}


In [ ]:
jobs = []

for method_key, method_name, train_function in TRAINING_1_VARIANTS:
    for run_number in range(1, NUM_REPEATS + 1):
        method_settings = deepcopy(TRAINING_1_SETTINGS[method_key])
        experiment_name = f"{method_key}_r{run_number}"

        jobs.append(
            make_training_job(
                experiment_name=experiment_name,
                train_function=train_function,
                training_seeds=TRAINING_SEEDS[f"r{run_number}"],
                test_seeds=TEST_SEEDS,
                settings=method_settings,
                reverse_positions=method_settings.add_reversed_positions,
                results_dir=RESULTS_DIR,
            )
        )

run_training_jobs(jobs)


In [ ]:
n_mean_reward_train_basic_results = []

for method_key, method_name, train_function in TRAINING_1_VARIANTS:
    n_value = TRAINING_1_N_VALUES[method_key]
    n_mean_reward_train_basic_results.append(
        collect_metric_results(
            method_key=method_key,
            method_name=f"n={n_value}",
            calculate_function=calculate_mean_mean_reward_train_basic,
        )
    )

n_mean_reward_train_basic_results_sorted = sort_groups_by_mean_desc(
    n_mean_reward_train_basic_results
)
best_n_step_name = n_mean_reward_train_basic_results_sorted[0][0]
best_n_step = int(best_n_step_name.replace("n=", ""))
best_n_step_mean = sum(n_mean_reward_train_basic_results_sorted[0][1]) / len(
    n_mean_reward_train_basic_results_sorted[0][1]
)

nstep_dqn_mean_reward_train_basic_results = [
    result
    for result in n_mean_reward_train_basic_results
    if result[0] != "n=1"
]
nstep_dqn_mean_reward_train_basic_results_sorted = sort_groups_by_mean_desc(
    nstep_dqn_mean_reward_train_basic_results
)
best_nstep_dqn_name = nstep_dqn_mean_reward_train_basic_results_sorted[0][0]
best_nstep_dqn = int(best_nstep_dqn_name.replace("n=", ""))
best_nstep_dqn_mean = sum(nstep_dqn_mean_reward_train_basic_results_sorted[0][1]) / len(
    nstep_dqn_mean_reward_train_basic_results_sorted[0][1]
)
best_nstep_dqn_method_key = f"nstep_dqn_n{best_nstep_dqn}"
best_nstep_dqn_method_name = f"N-step DQN (n={best_nstep_dqn})"

n_settings = deepcopy(settings)
n_settings.n_step = best_n_step
n_settings.save(SETTINGS_DIR / "n_settings.pickle")

training_compare_table(
    groups=n_mean_reward_train_basic_results,
    subtitle=(
        "Dobór n: średnia wartość średniej nagrody, "
        "środowisko treningowe, model bazowy"
    ),
)

print(
    f"Najlepsze n ogólnie: {best_n_step} "
    f"(średnia wartość średniej nagrody train/basic = {best_n_step_mean:.7g})"
)
print(
    f"Najlepszy N-step DQN: n={best_nstep_dqn} "
    f"(średnia wartość średniej nagrody train/basic = {best_nstep_dqn_mean:.7g})"
)


In [ ]:
rainbow_dqn_settings = GlobalSettings.load(SETTINGS_DIR / "n_settings.pickle")
print(f"Rainbow DQN będzie trenowany z n={rainbow_dqn_settings.n_step}.")


<a id="eksperyment-2"></a>
### Eksperyment 2 - Porównanie metod RL


#### Opis:
W drugim etapie trenowane są pozostałe metody RL. Wszystkie metody korzystają z głównego obiektu `settings` zdefiniowanego na początku notebooka, z wyjątkiem Rainbow DQN, który korzysta z zapisanego ustawienia najlepszego `n` wybranego w eksperymencie 1.


In [ ]:
TRAINING_2_VARIANTS = [
    ("double_dqn", "Double DQN", train_double_dqn),
    ("dueling_dqn", "Dueling DQN", train_dueling_dqn),
    ("per_dqn", "PER DQN", train_per_dqn),
    ("noisy_dqn", "Noisy DQN", train_noisy_dqn),
    ("distributional_dqn", "Distributional DQN", train_distributional_dqn),
    ("rainbow_dqn", "Rainbow DQN", train_rainbow_dqn),
    ("reinforce", "REINFORCE", train_reinforce),
    ("a2c", "A2C", train_a2c),
    ("ppo", "PPO", train_ppo),
]

TRAINING_COMPARE_VARIANTS = [
    ("dqn", "DQN", train_dqn),
    (best_nstep_dqn_method_key, best_nstep_dqn_method_name, train_nstep_dqn),
]

TRAINING_COMPARE_VARIANTS.extend(TRAINING_2_VARIANTS)


In [ ]:
jobs = []

for method_key, method_name, train_function in TRAINING_2_VARIANTS:
    for run_number in range(1, NUM_REPEATS + 1):
        method_settings = deepcopy(
            rainbow_dqn_settings if method_key == "rainbow_dqn" else settings
        )
        experiment_name = f"{method_key}_r{run_number}"

        jobs.append(
            make_training_job(
                experiment_name=experiment_name,
                train_function=train_function,
                training_seeds=TRAINING_SEEDS[f"r{run_number}"],
                test_seeds=TEST_SEEDS,
                settings=method_settings,
                reverse_positions=method_settings.add_reversed_positions,
                results_dir=RESULTS_DIR,
            )
        )

run_training_jobs(jobs)


In [ ]:
TRAINING_COMPARE_METRICS = [
    (
        "Średni odsetek wygranej, środowisko treningowe, model bazowy",
        calculate_mean_winrate_train_basic,
    ),
    (
        "Średni odsetek wygranej, środowisko testowe, model bazowy",
        calculate_mean_winrate_test_basic,
    ),
    (
        "Średni odsetek wygranej, środowisko treningowe, model poprzedni",
        calculate_mean_winrate_train_previous,
    ),
    (
        "Średni odsetek wygranej, środowisko testowe, model poprzedni",
        calculate_mean_winrate_test_previous,
    ),
    (
        "Średnia wartość średniej długości epizodu, środowisko treningowe, model bazowy",
        calculate_mean_episode_length_train_basic,
    ),
    (
        "Średnia wartość średniej długości epizodu, środowisko testowe, model bazowy",
        calculate_mean_episode_length_test_basic,
    ),
    (
        "Średnia wartość średniej długości epizodu, środowisko treningowe, model poprzedni",
        calculate_mean_episode_length_train_previous,
    ),
    (
        "Średnia wartość średniej długości epizodu, środowisko testowe, model poprzedni",
        calculate_mean_episode_length_test_previous,
    ),
]

for metric_title, calculate_function in TRAINING_COMPARE_METRICS:
    metric_results = build_metric_results(TRAINING_COMPARE_VARIANTS, calculate_function)
    training_compare_table(
        groups=metric_results,
        subtitle=metric_title,
    )


In [ ]:
auc_reward_test_basic_results = build_metric_results(
    TRAINING_COMPARE_VARIANTS,
    calculate_auc_reward_test_basic,
)

training_compare_table(
    groups=auc_reward_test_basic_results,
    subtitle=(
        "Znormalizowane AUC średniej łącznej nagrody, "
        "środowisko testowe, model bazowy"
    ),
)


In [ ]:
kruskal_status, kruskal_stat, kruskal_p = test_kruskal(
    groups=auc_reward_test_basic_results,
    significant_threshold=0.05,
    very_significant_threshold=0.01,
    print_result=True,
)

if kruskal_status != SignificantStatus.NOT_SIGNIFICANT:
    test_mannwhitneyu(
        groups=auc_reward_test_basic_results,
        significant_threshold=0.05,
        very_significant_threshold=0.01,
        print_result=True,
        subtitle=(
            "Znormalizowane AUC średniej łącznej nagrody, "
            "środowisko testowe, model bazowy"
        ),
    )
else:
    print(
        "Test Kruskala-Wallisa nie wykazał istotnych różnic statystycznych, "
        "dlatego zaniechano wykonania testu Manna-Whitneya U."
    )


#### Przykładowe przebiegi treningu


In [ ]:
TRAINING_SUMMARY_VARIANTS = [
    *TRAINING_1_VARIANTS,
    *TRAINING_2_VARIANTS,
]

for method_key, method_name, train_function in TRAINING_SUMMARY_VARIANTS:
    show_training_summary(
        global_fig_title=f"Przykładowy przebieg treningu - {method_name}",
        training_results=load_results(f"{method_key}_r1"),
    )


<a id="eksperyment-3"></a>
### Eksperyment 3 - Porównanie algorytmów poprzez pojedynki otrzymanych modeli


#### Opis:
W tym etapie wytrenowane modele rywalizują między sobą w parach. Pojedynki są wykonywane zarówno na środowiskach treningowych, jak i testowych.


In [ ]:
NUM_DUELS_TRAINING = 20
NUM_DUELS_TEST = 20


In [ ]:
def get_last_model_path(experiment_name: str) -> str:
    df = load_results(experiment_name)
    return df.iloc[-1]["model_path"]


In [ ]:
base_agent = DecisionTreeAgent(settings=settings)

dqn_agent = AgentNN.load(
    get_last_model_path("dqn_r1")
)

best_nstep_agent = AgentNN.load(
    get_last_model_path(f"nstep_dqn_n{best_nstep_dqn}_r1")
)

double_dqn_agent = AgentNN.load(
    get_last_model_path("double_dqn_r1")
)

dueling_dqn_agent = AgentNN.load(
    get_last_model_path("dueling_dqn_r1")
)

per_dqn_agent = AgentNN.load(
    get_last_model_path("per_dqn_r1")
)

noisy_dqn_agent = AgentNN.load(
    get_last_model_path("noisy_dqn_r1")
)

distributional_dqn_agent = AgentNN.load(
    get_last_model_path("distributional_dqn_r1")
)

rainbow_dqn_agent = AgentNN.load(
    get_last_model_path("rainbow_dqn_r1")
)

reinforce_agent = AgentNN.load(
    get_last_model_path("reinforce_r1")
)

a2c_agent = AgentNN.load(
    get_last_model_path("a2c_r1")
)

ppo_agent = AgentNN.load(
    get_last_model_path("ppo_r1")
)

E5_AGENTS = [
    ("Agent bazowy", base_agent),
    ("DQN", dqn_agent),
    (f"N-step DQN (n={best_nstep_dqn})", best_nstep_agent),
    ("Double DQN", double_dqn_agent),
    ("Dueling DQN", dueling_dqn_agent),
    ("PER DQN", per_dqn_agent),
    ("Noisy DQN", noisy_dqn_agent),
    ("Distributional DQN", distributional_dqn_agent),
    ("Rainbow DQN", rainbow_dqn_agent),
    ("REINFORCE", reinforce_agent),
    ("A2C", a2c_agent),
    ("PPO", ppo_agent),
]


In [ ]:
duel_results = []

for agent1_tuple, agent2_tuple in combinations(E5_AGENTS, 2):
    df_agent1, df_agent2 = duels(
        agent1=agent1_tuple,
        agent2=agent2_tuple,
        training_seeds=DUEL_SEEDS,
        test_seeds=DUEL_SEEDS,
        settings=settings,
        reverse_positions=settings.add_reversed_positions,
        num_duels_training=NUM_DUELS_TRAINING,
        num_duels_test=NUM_DUELS_TEST,
    )

    duel_results.append(df_agent1)
    duel_results.append(df_agent2)

all_duels_df = pd.concat(duel_results, ignore_index=True)
save_results(all_duels_df, "e5_duels")


In [ ]:
all_duels_df = load_results("e5_duels")


In [ ]:
show_heatmap_winrate_train(all_duels_df)
show_heatmap_winrate_test(all_duels_df)


In [ ]:
show_heatmap_winrate_diff(all_duels_df)


In [ ]:
show_heatmap_episode_train(all_duels_df)
show_heatmap_episode_test(all_duels_df)


<a id="filmy-pojedynkow"></a>
## 5. Przykładowe pojedynki (filmy)


In [ ]:
from IPython.display import Video, display, Markdown

from src.env_gui.renderer import render_duel

VIDEO_OUTPUT_DIR = Path("outputs/videos")
VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def get_video_model_path(experiment_name: str) -> str:
    df = load_results(experiment_name)
    return df.iloc[-1]["model_path"]

video_base_agent = DecisionTreeAgent(settings=settings)
video_dqn_agent = AgentNN.load(get_video_model_path("dqn_r1"))
video_rainbow_dqn_agent = AgentNN.load(get_video_model_path("rainbow_dqn_r1"))
video_ppo_agent = AgentNN.load(get_video_model_path("ppo_r1"))
video_reinforce_agent = AgentNN.load(get_video_model_path("reinforce_r1"))

EXAMPLE_DUEL_VIDEOS = [
    {
        "title": "DQN vs Rainbow DQN",
        "filename": "dqn_vs_rainbw_dqn.mp4",
        "agent1": ("DQN", video_dqn_agent),
        "agent2": ("Rainbow DQN", video_rainbow_dqn_agent),
        "seed": "video_dqn_vs_rainbow_dqn",
    },
    {
        "title": "DQN vs DecisionTree",
        "filename": "dqn_vs_decision_tree.mp4",
        "agent1": ("DQN", video_dqn_agent),
        "agent2": ("DecisionTree", video_base_agent),
        "seed": "video_dqn_vs_decision_tree",
    },
    {
        "title": "DecisionTree vs DecisionTree",
        "filename": "decision_tree_vs_decision_tree.mp4",
        "agent1": ("DecisionTree", video_base_agent),
        "agent2": ("DecisionTree", DecisionTreeAgent(settings=settings)),
        "seed": "video_decision_tree_vs_decision_tree",
    },
    {
        "title": "Rainbow DQN vs PPO",
        "filename": "rainbow_dqn_vs_ppo.mp4",
        "agent1": ("Rainbow DQN", video_rainbow_dqn_agent),
        "agent2": ("PPO", video_ppo_agent),
        "seed": "video_rainbow_dqn_vs_ppo",
    },
    {
        "title": "PPO vs REINFORCE",
        "filename": "ppo_vs_reinforce.mp4",
        "agent1": ("PPO", video_ppo_agent),
        "agent2": ("REINFORCE", video_reinforce_agent),
        "seed": "video_ppo_vs_reinforce",
    },
]

for duel_video in EXAMPLE_DUEL_VIDEOS:
    output_path = VIDEO_OUTPUT_DIR / duel_video["filename"]

    if not output_path.exists():
        render_duel(
            agent1=duel_video["agent1"],
            agent2=duel_video["agent2"],
            seed=duel_video["seed"],
            settings=settings,
            output_path=str(output_path),
            reverse_positions=False,
        )

    display(Markdown(f"#### {duel_video['title']}"))
    display(Video(str(output_path), embed=False, html_attributes="controls"))
